# Assignment 3 — Forward Propagation, Backpropagation and Hyperparameters

**Platform:** Google Colab &nbsp;|&nbsp; **Suggested runtime:** CPU  
**How to use:** Run the cells from top to bottom. Change the small experiment
constants when more training time is available.

This workbook is written as a compact college assignment: it explains the
problem, implements the method, evaluates the result, and records the main
observations.


## Problem and theory

Forward propagation calculates predictions layer by layer. The loss
measures prediction error. During backpropagation, automatic
differentiation calculates the gradient of the loss with respect to
every trainable weight, and the optimizer updates those weights.

We use a two-moons classification problem and compare learning rates
and epoch counts. All runs start from the same seed for a fairer test.


In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

SEED = 42
X, y = make_moons(n_samples=1200, noise=0.22, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X.astype("float32"), y.astype("float32"), test_size=0.25,
    random_state=SEED, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

plt.figure(figsize=(6, 4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=12)
plt.title("Two-moons training data"); plt.show()


## One explicit forward/backpropagation step


In [ ]:
tf.keras.utils.set_random_seed(SEED)
demo_model = tf.keras.Sequential([
    tf.keras.layers.Input((2,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
optimizer = tf.keras.optimizers.SGD(learning_rate=0.05)
loss_fn = tf.keras.losses.BinaryCrossentropy()

xb = tf.convert_to_tensor(X_train[:32])
yb = tf.convert_to_tensor(y_train[:32, None])
before = [w.numpy().copy() for w in demo_model.trainable_weights]

with tf.GradientTape() as tape:
    y_probability = demo_model(xb, training=True)       # forward pass
    batch_loss = loss_fn(yb, y_probability)
gradients = tape.gradient(batch_loss, demo_model.trainable_weights)  # backprop
optimizer.apply_gradients(zip(gradients, demo_model.trainable_weights))

weight_change = np.mean([np.mean(np.abs(a - w.numpy()))
                         for a, w in zip(before, demo_model.trainable_weights)])
print(f"Batch loss: {batch_loss.numpy():.4f}")
print(f"Mean absolute weight update: {weight_change:.6f}")


## Controlled experiment


In [ ]:
def build_model(seed=SEED):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    return tf.keras.Sequential([
        tf.keras.layers.Input((2,)),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(8, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])

def run_experiment(learning_rate, epochs):
    model = build_model()
    model.compile(
        optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
        loss="binary_crossentropy", metrics=["accuracy"]
    )
    history = model.fit(
        X_train, y_train, validation_split=0.20, epochs=epochs,
        batch_size=32, verbose=0, shuffle=True
    )
    prediction = (model.predict(X_test, verbose=0).ravel() >= 0.5).astype(int)
    return accuracy_score(y_test, prediction), history.history

learning_rates = [0.001, 0.01, 0.1]
epoch_counts = [20, 60, 120]
rows, histories = [], {}

for lr in learning_rates:
    for epochs in epoch_counts:
        accuracy, hist = run_experiment(lr, epochs)
        rows.append({"learning_rate": lr, "epochs": epochs,
                     "test_accuracy": accuracy,
                     "final_val_loss": hist["val_loss"][-1]})
        histories[(lr, epochs)] = hist

results = pd.DataFrame(rows)
display(results.sort_values("test_accuracy", ascending=False).round(4))


In [ ]:
pivot = results.pivot(index="learning_rate", columns="epochs", values="test_accuracy")
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu", vmin=0.5, vmax=1.0)
plt.title("Test accuracy by learning rate and epochs")
plt.show()

plt.figure(figsize=(8, 4))
for lr in learning_rates:
    h = histories[(lr, max(epoch_counts))]
    plt.plot(h["val_loss"], label=f"lr={lr}")
plt.xlabel("Epoch"); plt.ylabel("Validation loss")
plt.title("Learning-rate comparison"); plt.legend(); plt.show()


## Analysis guide

- A very small learning rate changes weights slowly and may need more epochs.
- A suitable rate converges quickly and steadily.
- An excessive rate can overshoot a good solution or make loss unstable.
- More epochs help until convergence; after that they add cost and can overfit.

Select the setting with strong test accuracy and stable validation loss—not
simply the longest run.


## Conclusion

The experiment above provides a complete training and evaluation workflow. The
printed metrics and plots are the result for the current run and should be used
to identify the strongest behaviour, the main limitation, and one justified
improvement. Exact values may vary slightly because neural-network training is
stochastic.
